In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import warnings
warnings.filterwarnings('ignore')


spark = SparkSession.builder \
    .appName("Clustering_Project") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()


# hide warns messages
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.json("cleaned_meta_beauty.json")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 16:31:59 WARN Utils: Your hostname, Pangkaews-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.32.148.207 instead (on interface en0)
26/03/11 16:31:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 16:31:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [2]:
df.select("title","description","features_string").show(5)
total_count = df.count()
feature_nan = df.filter(F.col("features_string").isNull() | (F.col("features_string")=="")).count()
descrip_nan = df.filter(F.col("description").isNull() | (F.col("features_string")=="")).count()
print(f"feature missing value : {feature_nan/total_count}")
print(f"description missing value : {descrip_nan/total_count}")

+--------------------+--------------------+--------------------+
|               title|         description|     features_string|
+--------------------+--------------------+--------------------+
|howard lc008 leat...|                    |                    |
|yes tomato detoxi...|                    |                    |
|eye patch black a...|                    |                    |
|tattoo eyebrow st...|                    |                    |
|precision plunger...|precision plunger...|material 304 stai...|
+--------------------+--------------------+--------------------+
only showing top 5 rows
feature missing value : 0.8456967759126033
description missing value : 0.8456967759126033


In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, PCA, VectorAssembler
from pyspark.sql.functions import col, concat_ws

df = df.withColumn("text_combined", concat_ws(" ", col("title"), col("description"), col("features_string")))

tokenizer = Tokenizer(inputCol="text_combined", outputCol="words")
custom_stopwords = ["oz", "pack","0", "1", "2", "3", "4", "5", "bottle", "container","inch","x","set","ounce"]
default_stopwords = StopWordsRemover.loadDefaultStopWords("english")
all_stopwords = default_stopwords + custom_stopwords
#all_stopwords = default_stopwords
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words", stopWords=all_stopwords)

hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="tfidf_features")

pca = PCA(k=100, inputCol="tfidf_features", outputCol="pca_features")

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, pca])
model = pipeline.fit(df)
result = model.transform(df)

result.select("parent_asin", "pca_features").show(5)

+-----------+--------------------+
|parent_asin|        pca_features|
+-----------+--------------------+
| B01CUPMQZE|[-0.3307079071947...|
| B076WQZGPM|[-1.6818040507853...|
| B000B658RI|[-0.7513748920672...|
| B088FKY3VD|[-2.3829856595329...|
| B07NGFDN6G|[-9.3710958045515...|
+-----------+--------------------+
only showing top 5 rows


In [4]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

def find_k(df_result):
    for k in range(2, 11):
        kmeans = KMeans(featuresCol="pca_features", k=k,seed=42)
        model = kmeans.fit(df_result)
        predictions = model.transform(df_result)
        evaluator = ClusteringEvaluator(featuresCol="pca_features")
        silhouette = evaluator.evaluate(predictions)
        print(f"With K={k}, Silhouette Score = {silhouette}")
    return 0

print(find_k(result))

With K=2, Silhouette Score = 0.8431047882123307


With K=3, Silhouette Score = 0.5088074007965268


With K=4, Silhouette Score = 0.8209718565714539


With K=5, Silhouette Score = 0.8110425530525276


With K=6, Silhouette Score = 0.8164589915116299


With K=7, Silhouette Score = 0.27690925965105656


With K=8, Silhouette Score = 0.2271560320925864


With K=9, Silhouette Score = 0.23922788054664273


With K=10, Silhouette Score = 0.312636061357094
0


In [5]:
# k = 2
def fit_model(df_result, selected_k):
    kmeans_model = KMeans(featuresCol="pca_features", k=selected_k,seed=42)
    model = kmeans_model.fit(df_result)
    predictions = model.transform(df_result)
    return model,predictions

model_r1, prediction_r1 = fit_model(result,2)



In [6]:
from pyspark.sql import functions as F
prediction_r1.groupBy("prediction").agg(F.count("*")).show()

+----------+--------+
|prediction|count(1)|
+----------+--------+
|         1|    8172|
|         0|  104418|
+----------+--------+



In [7]:
from pyspark.sql import functions as F

df_with_clusters = df.join(prediction_r1.select("parent_asin", "prediction"), on="parent_asin", how="inner")

# re-cluster
# spit data
df_0 = df_with_clusters.filter(F.col("prediction") == 0)
df_0 = df_0.drop("prediction")
df_1 = df_with_clusters.filter(F.col("prediction") == 1)
df_1 = df_1.drop("prediction")

# PCA
model_r2_c0 = pipeline.fit(df_0)
model_r2_c1 = pipeline.fit(df_1)
result_r2_c0 = model_r2_c0.transform(df_0)
result_r2_c1 = model_r2_c1.transform(df_1)

print("Find K for c0\n")
print(find_k(result_r2_c0))

print("Find K for c1\n")
print(find_k(result_r2_c1))




Find K for c0



With K=2, Silhouette Score = 0.17017408855722893


With K=3, Silhouette Score = 0.2871289189306317


With K=4, Silhouette Score = 0.1538468661058692


With K=5, Silhouette Score = 0.20292894034295791


With K=6, Silhouette Score = 0.289579378608495
With K=7, Silhouette Score = 0.2961269177869652
With K=8, Silhouette Score = 0.25486019899678314


With K=9, Silhouette Score = 0.21962556874577738


With K=10, Silhouette Score = 0.21630797896171003
0
Find K for c1

With K=2, Silhouette Score = 0.48081967509363616
With K=3, Silhouette Score = 0.3576626627429717
With K=4, Silhouette Score = 0.35552892124964003
With K=5, Silhouette Score = 0.12324204580116835
With K=6, Silhouette Score = 0.39918085872699477
With K=7, Silhouette Score = 0.3816453054319033
With K=8, Silhouette Score = 0.19656528904034393
With K=9, Silhouette Score = 0.2037970830956573
With K=10, Silhouette Score = 0.18917810845924735
0


In [8]:
# k=7 for cluster0 and k=2 for cluster1
model_r2_c0, prediction_r2_c0 = fit_model(result_r2_c0,7)
model_r2_c1, prediction_r2_c1 = fit_model(result_r2_c1,2)

df_r2_c0 = df_0.join(prediction_r2_c0.select("parent_asin", "prediction"), on="parent_asin", how="inner")
df_r2_c1 = df_1.join(prediction_r2_c1.select("parent_asin", "prediction"), on="parent_asin", how="inner")

In [9]:
from pyspark.ml.feature import Tokenizer
from pyspark.sql import functions as F
from pyspark.ml.feature import StopWordsRemover

def extract_keyword(df,cluster_id):
    tokenizer_test = Tokenizer(inputCol="title", outputCol="words")
    words_df_test = tokenizer_test.transform(df)
    
    custom_stopwords = ["oz", "pack","0", "1", "2", "3", "4", "5","6","8", "bottle", "container","inch","x","set","ounce","fl"]
    default_stopwords = StopWordsRemover.loadDefaultStopWords("english")
    all_stopwords = default_stopwords + custom_stopwords
    remover = StopWordsRemover(inputCol="words", outputCol="filtered", stopWords=all_stopwords)
    clean_df = remover.transform(words_df_test)
    
    clean_df.filter(F.col("prediction") == cluster_id) \
            .select(F.explode("filtered").alias("word")) \
            .groupBy("word").count() \
            .orderBy(F.col("count").desc()).show(10)
    return 0

print("top keywords from 1st round, cluster 0\n")
print(extract_keyword(df_with_clusters,0))

print("top keywords from 1st round, cluster 1\n")
print(extract_keyword(df_with_clusters,1))

k_c0 = 7
k_c1 = 2

for i in range(k_c0):
    print(f"top keywords from 2nd round, cluster0,{i}\n")
    print(extract_keyword(df_r2_c0,i))

for i in range(k_c1):
    print(f"top keywords from 2nd round, cluster1,{i}\n")
    print(extract_keyword(df_r2_c1,i))


top keywords from 1st round, cluster 0

+---------+-----+
|     word|count|
+---------+-----+
|     hair|75124|
|     nail|22032|
|    woman|22005|
|      wig|20463|
|extension|16336|
|    human|15640|
|  natural|13985|
|    black|13941|
|    color|12818|
|   bundle|10190|
+---------+-----+
only showing top 10 rows
0
top keywords from 1st round, cluster 1

+-------+-----+
|   word|count|
+-------+-----+
|   hair| 4492|
|   nail| 1808|
|  woman| 1706|
|  brush| 1390|
|   face| 1369|
|   skin| 1289|
|natural| 1197|
|    wig| 1162|
|   body|  909|
|  black|  845|
+-------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,0



+---------+-----+
|     word|count|
+---------+-----+
|     hair|62761|
|extension|14026|
|    woman|13793|
|    human|11486|
|   bundle|10004|
|    color| 9162|
|     clip| 9052|
|  natural| 8755|
|    black| 8744|
|     body| 7804|
+---------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,1



+----------+-----+
|      word|count|
+----------+-----+
|   eyebrow| 3177|
|       lip| 2019|
|waterproof|  972|
|      brow|  938|
|       kit|  876|
|    makeup|  872|
|   lasting|  795|
|      long|  766|
|  lipstick|  719|
|     stamp|  708|
+----------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,2

+---------+-----+
|     word|count|
+---------+-----+
|      wig|18693|
|     hair| 8435|
|    woman| 5198|
|     lace| 5065|
|    human| 4066|
|    black| 3524|
|    front| 3327|
|synthetic| 2873|
|    curly| 2385|
|  natural| 2369|
+---------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,3



+--------+-----+
|    word|count|
+--------+-----+
|    nail|18668|
|     art| 3605|
|     gel| 2841|
|  polish| 2548|
| acrylic| 2103|
|manicure| 1962|
|     tip| 1812|
|     kit| 1611|
|    fake| 1541|
|   false| 1503|
+--------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,4



+-------+-----+
|   word|count|
+-------+-----+
|   hair| 1317|
|   body|  750|
|  woman|  670|
|   skin|  520|
|    oil|  503|
|  black|  483|
|  cream|  446|
|natural|  406|
|   face|  372|
|  spray|  346|
+-------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,5



+----------+-----+
|      word|count|
+----------+-----+
|     brush| 6749|
|      hair| 2379|
|    makeup| 1659|
|      comb|  737|
|  cosmetic|  597|
|      tool|  579|
|  silicone|  573|
|      body|  505|
|foundation|  505|
|   bristle|  502|
+----------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster0,6



+---------+-----+
|     word|count|
+---------+-----+
|  eyelash| 6032|
|     lash| 5117|
|    false| 2255|
|extension| 1722|
|     pair| 1594|
|     mink| 1519|
|  natural| 1422|
|   volume| 1388|
|       3d| 1329|
| magnetic| 1315|
+---------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster1,0

+-------+-----+
|   word|count|
+-------+-----+
|   hair| 4169|
|   nail| 1753|
|  woman| 1671|
|   face| 1311|
|   skin| 1238|
|natural| 1167|
|    wig| 1162|
|   body|  840|
|  black|  823|
| roller|  816|
+-------+-----+
only showing top 10 rows
0
top keywords from 2nd round, cluster1,1

+--------+-----+
|    word|count|
+--------+-----+
|   brush|  870|
|     dog|  399|
|    hair|  323|
|     pet|  298|
|     cat|  276|
|grooming|  207|
|    comb|  202|
|cleaning|  117|
|    long|  116|
|    bath|  109|
+--------+-----+
only showing top 10 rows
0


In [10]:
# report
def profile_clusters(df, k_count, cluster_name_prefix):
    for i in range(k_count):
        print(f"\n{'='*30}")
        print(f"PROFILE: Cluster {cluster_name_prefix}.{i}")
        
        # size
        count = df.filter(F.col("prediction") == i).count()
        print(f"Size: {count} products")
        
        # defining Characteristics (Keywords)
        print("Characteristics (Top 5):")
        # Reuse your existing logic but just show top 5
        extract_keyword(df, i) # This will print the top 10 from your function
        
        # representative Examples
        print("Representative Examples:")
        df.filter(F.col("prediction") == i).select("title").show(2, truncate=False)

profile_clusters(df_r2_c0, 7, "0")
profile_clusters(df_r2_c1, 2, "1")


PROFILE: Cluster 0.0


Size: 77263 products
Characteristics (Top 5):


+---------+-----+
|     word|count|
+---------+-----+
|     hair|62761|
|extension|14026|
|    woman|13793|
|    human|11486|
|   bundle|10004|
|    color| 9162|
|     clip| 9052|
|  natural| 8755|
|    black| 8744|
|     body| 7804|
+---------+-----+
only showing top 10 rows
Representative Examples:


+--------------------------------------------------------------------------------------------------------------------------------------------------+
|title                                                                                                                                             |
+--------------------------------------------------------------------------------------------------------------------------------------------------+
|feiding liquid highlighter illuminator rose gold body shimmer oil spray face arm leg shimming glow lotion dark skin glitter body shimmer rose gold|
|intimate tropical strawberry shimmer mist                                                                                                         |
+--------------------------------------------------------------------------------------------------------------------------------------------------+
only showing top 2 rows

PROFILE: Cluster 0.1
Size: 2178 products
Characteristics (Top 5):


+----------+-----+
|      word|count|
+----------+-----+
|   eyebrow| 3177|
|       lip| 2019|
|waterproof|  972|
|      brow|  938|
|       kit|  876|
|    makeup|  872|
|   lasting|  795|
|      long|  766|
|  lipstick|  719|
|     stamp|  708|
+----------+-----+
only showing top 10 rows
Representative Examples:


...

+-------+-----+
|   word|count|
+-------+-----+
|   hair| 1317|
|   body|  750|
|  woman|  670|
|   skin|  520|
|    oil|  503|
|  black|  483|
|  cream|  446|
|natural|  406|
|   face|  372|
|  spray|  346|
+-------+-----+
only showing top 10 rows
Representative Examples:
+--------------------------------------------------------------------------+
|title                                                                     |
+--------------------------------------------------------------------------+
|yardley poise deodorant refreshing body spray 150ml pack 2 herbalstore_247|
|axe styling natural look conditioning cream 2 64 ounce pack 10            |
+--------------------------------------------------------------------------+
only showing top 2 rows

PROFILE: Cluster 0.5
Size: 3093 products
Characteristics (Top 5):


+----------+-----+
|      word|count|
+----------+-----+
|     brush| 6749|
|      hair| 2379|
|    makeup| 1659|
|      comb|  737|
|  cosmetic|  597|
|      tool|  579|
|  silicone|  573|
|      body|  505|
|foundation|  505|
|   bristle|  502|
+----------+-----+
only showing top 10 rows
Representative Examples:


+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|title                                                                                                                                                                   |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|pet grooming glove gentle deshedding brush glove efficient pet hair remover mitt enhanced five finger design perfect dog cat long short fur 1 pair                      |
|easace razor extension shaver handle adjustable length bend razor body shaver handle groomer body brush handle wet dry brushing multiple function razor shaver back blue|
+------------------------------------------------------------------------------------------------------------------------------------------------

+---------+-----+
|     word|count|
+---------+-----+
|  eyelash| 6032|
|     lash| 5117|
|    false| 2255|
|extension| 1722|
|     pair| 1594|
|     mink| 1519|
|  natural| 1422|
|   volume| 1388|
|       3d| 1329|
| magnetic| 1315|
+---------+-----+
only showing top 10 rows
Representative Examples:


+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|title                                                                                                                                                      |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|wimzy creation updated 3d 6d magnetic eyelash eyeliner set 2 tube magnetic eyeliner 10 pair magnetic eyelash kit natural look reusable false lash glue need|
|lagee premium flat ellipse eyelash extension matte black 0 12mm d curl 12mm mixed faux mink lash individual classic split tip professional lash supply     |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
only showing top 2 rows

PROFILE: Cluster 1.0
Size: 

In [11]:
# evaluation

from pyspark.ml.evaluation import ClusteringEvaluator

def get_metrics(model, predictions, features_col="pca_features"):
    evaluator = ClusteringEvaluator(featuresCol=features_col, metricName="silhouette")
    silhouette = evaluator.evaluate(predictions)
    wssse = model.summary.trainingCost
    
    return silhouette, wssse

# Round 1 Evaluation
s1, w1 = get_metrics(model_r1, prediction_r1)
print(f"Round 1 (K=2): Silhouette={s1:.4f}, WSSSE={w1:.2f}")

# Round 2 - Cluster 0 Evaluation
s2_c0, w2_c0 = get_metrics(model_r2_c0, prediction_r2_c0)
print(f"Round 2 - Cluster 0 (K=8): Silhouette={s2_c0:.4f}, WSSSE={w2_c0:.2f}")

# Round 2 - Cluster 1 Evaluation
s2_c1, w2_c1 = get_metrics(model_r2_c1, prediction_r2_c1)
print(f"Round 2 - Cluster 1 (K=3): Silhouette={s2_c1:.4f}, WSSSE={w2_c1:.2f}")

Round 1 (K=2): Silhouette=0.8431, WSSSE=42983962.20
Round 2 - Cluster 0 (K=8): Silhouette=0.2961, WSSSE=13523255.64
Round 2 - Cluster 1 (K=3): Silhouette=0.4808, WSSSE=11005079.70


In [13]:
from pyspark.ml.evaluation import ClusteringEvaluator

# Define an evaluator
evaluator = ClusteringEvaluator(featuresCol="pca_features", metricName="silhouette")

def get_silhouette_score(df_with_preds):
    # This assumes 'df_with_preds' has a 'pca_features' column and a 'prediction' column
    return evaluator.evaluate(df_with_preds)

# Calculate scores for each branch of your hierarchy
score_c0 = get_silhouette_score(prediction_r2_c0)
score_c1 = get_silhouette_score(prediction_r2_c1)

# Weighted Average Silhouette Score for the whole model
total_points = df_r2_c0.count() + df_r2_c1.count()
weighted_silhouette = (
    (df_r2_c0.count() * score_c0) + (df_r2_c1.count() * score_c1)
) / total_points

print(f"Hierarchical Model Global Silhouette Score: {weighted_silhouette:.4f}")

Hierarchical Model Global Silhouette Score: 0.3095
